In [1]:
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from datetime import datetime
import requests
import json
import time

def train_model():
    # Load your model and data here
    data = pd.read_csv('waterfall_data.csv')
    data.columns = data.columns.str.strip()  # Remove leading/trailing spaces from column names
    
    # Check if 'Weather Description' is present in the dataset
    if 'Weather Description' not in data.columns:
        raise KeyError("Column 'Weather Description' not found in the CSV data")
    
    # Map 'Weather Description' categories to numeric values for model training
    weather_map = {
        'light rain': 0, 'moderate rain': 1, 'heavy rain': 2, 'overcast clouds': 3, 'shower rain': 4,
        'drizzle': 5, 'snow': 6, 'mist': 7, 'broken clouds': 8, 'clear sky': 9, 'few clouds': 10, 'thunderstorm': 11
    }
    data['Weather Description'] = data['Weather Description'].map(weather_map)
    
    # Drop rows with missing values in 'Weather Description'
    data = data.dropna(subset=['Weather Description'])

    # Use relevant features for prediction
    X = data[['Humidity (%)', 'Temperature (°C)', 'Wind Speed (m/s)']]  # Features
    y = data['Weather Description']  # Target

    # Scale the features using StandardScaler
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    
    # Train the model with scaled data
    model = LogisticRegression(solver='lbfgs', max_iter=500)
    model.fit(X_scaled, y)  # Train the model
    return model, weather_map, scaler

def get_current_day_and_date():
    current_date = datetime.now()
    day_of_week = current_date.strftime('%A')
    date_str = current_date.strftime('%Y-%m-%d')
    return day_of_week, date_str

def get_current_time():
    current_time = datetime.now().strftime('%H:%M:%S')  # Get current time in HH:MM:SS format
    return current_time

def get_real_time_weather(api_key, city_name):
    url = f'http://api.openweathermap.org/data/2.5/weather?q={city_name}&appid={api_key}&units=metric'
    response = requests.get(url)
    data = response.json()
    if response.status_code == 200:
        humidity = data['main']['humidity']
        temperature = data['main']['temp']
        wind_speed = data['wind']['speed']
        weather_conditions = data['weather'][0]['description']  # Fetch weather description
        return humidity, temperature, wind_speed, weather_conditions
    else:
        print(f"Error fetching data: {data.get('message', 'Unknown error')}")
        return None, None, None, None

def predict_rainfall_and_flood(humidity, temperature, wind_speed, weather_conditions):
    # Normalize the weather description to lowercase for consistency
    rainfall_description = weather_conditions.lower()
    
    # Flood risk logic based on rainfall description
    if 'heavy rain' in rainfall_description or 'very heavy rain' in rainfall_description:
        flood_risk = 'Yes'
    elif 'thunderstorm' in rainfall_description or 'moderate rain' in rainfall_description or 'light rain' in rainfall_description:
        flood_risk = 'Unpredictable'
    else:
        flood_risk = 'No'
    
    return rainfall_description, flood_risk

def main():
    api_key = '616986177c8131321d11803ed44df570'
    city_name = 'Thenkasi'
    district_name = 'Thenkasi District'
    waterfall_name = 'Kutralam Falls'
    
    model, weather_map, scaler = train_model()  # Train the model on the CSV data
    
    # Run indefinitely and send data every 60 seconds (1 minute)
    while True:
        # Fetch real-time weather data from the API
        humidity, temperature, wind_speed, weather_conditions = get_real_time_weather(api_key, city_name)
        
        if humidity is not None:
            # Predict the rainfall category and flood risk
            predicted_rainfall, flood_risk = predict_rainfall_and_flood(humidity, temperature, wind_speed, weather_conditions)
            day, date = get_current_day_and_date()
            current_time = get_current_time()  # Get the current time
            
            # Prepare the data to send to Firebase
            data_to_send = {
                'date': date,
                'day': day,
                'time': current_time,
                'districtName': district_name,
                'waterfallName': waterfall_name,
                'temperature': f"{temperature}°C",
                'rainfallRate': predicted_rainfall,
                'floodRisk': flood_risk
            }
            
            # Send data to Firebase and use a unique key (timestamp) for overwriting previous data
            firebase_url ="https://my-water-flow-project-f1f53-default-rtdb.firebaseio.com/predictionData.json"

            response = requests.put(firebase_url, data=json.dumps(data_to_send))  # Using PUT to overwrite previous data
            
            if response.status_code == 200:
                print("Prediction, flood risk, time, and temperature successfully sent to Firebase")
            else:
                print(f"Error sending data to Firebase: {response.status_code}")
        
        # Wait for 1 minute before running again
        time.sleep(60)

if __name__ == '__main__':
    main()

KeyError: "Column 'Weather Description' not found in the CSV data"